<a href="https://colab.research.google.com/github/anfeng649-png/macro-econ/blob/main/oecd_growth_accounting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
from google.colab import files
uploaded = files.upload()
import pandas as pd

file_path = "/content/pwt90.xlsx"
df = pd.read_excel(file_path, sheet_name="Data")

print(df.head())
print(df.columns.tolist())
cols_needed = ["country", "year", "rgdpna", "rkna", "emp", "labsh"]
df = df[cols_needed].copy()

print(df.head())
print(df.shape)
print(df.isna().sum())
oecd_countries = [
    "Australia", "Austria", "Belgium", "Canada", "Denmark",
    "Finland", "France", "Germany", "Greece", "Iceland",
    "Ireland", "Italy", "Japan", "Netherlands", "New Zealand",
    "Norway", "Portugal", "Spain", "Sweden", "Switzerland",
    "United Kingdom", "United States"
]

df_oecd = df[df["country"].isin(oecd_countries)].copy()

print(df_oecd["country"].unique())
print(len(df_oecd["country"].unique()))
df_oecd = df_oecd[df_oecd["year"].between(1960, 2000)].copy()

print(df_oecd.head())
print(df_oecd.shape)
print(df_oecd["year"].min(), df_oecd["year"].max())
df_oecd = df_oecd.dropna(subset=["rgdpna", "rkna", "emp", "labsh"]).copy()

print(df_oecd.head())
print(df_oecd.shape)
print(df_oecd.isna().sum())
df_oecd["alpha_k"] = 1 - df_oecd["labsh"]

print(df_oecd[["country", "year", "labsh", "alpha_k"]].head())
df_oecd = df_oecd.sort_values(["country", "year"]).copy()

print(df_oecd[["country", "year"]].head(10))
import numpy as np

df_oecd["gY"] = df_oecd.groupby("country")["rgdpna"].transform(lambda x: np.log(x).diff())

print(df_oecd[["country", "year", "rgdpna", "gY"]].head(10))
df_oecd["gK"] = df_oecd.groupby("country")["rkna"].transform(lambda x: np.log(x).diff())

print(df_oecd[["country", "year", "rkna", "gK"]].head(10))
df_oecd["gL"] = df_oecd.groupby("country")["emp"].transform(lambda x: np.log(x).diff())

print(df_oecd[["country", "year", "emp", "gL"]].head(10))
df_oecd["income_growth"] = df_oecd["gY"] - df_oecd["gL"]

print(df_oecd[["country", "year", "gY", "gL", "income_growth"]].head(10))
df_oecd["capital_deepening"] = df_oecd["alpha_k"] * (df_oecd["gK"] - df_oecd["gL"])

print(df_oecd[["country", "year", "alpha_k", "gK", "gL", "capital_deepening"]].head(10))
df_oecd["tfp_growth"] = df_oecd["income_growth"] - df_oecd["capital_deepening"]

print(df_oecd[["country", "year", "income_growth", "capital_deepening", "tfp_growth"]].head(10))
df_growth = df_oecd.dropna(subset=["income_growth", "capital_deepening", "tfp_growth"]).copy()

print(df_growth[["country", "year", "income_growth", "capital_deepening", "tfp_growth"]].head(10))
print(df_growth.shape)
result = df_growth.groupby("country").agg({
    "income_growth": "mean",
    "tfp_growth": "mean",
    "capital_deepening": "mean",
    "alpha_k": "mean"
}).reset_index()

print(result.head())
print(result.shape)
result["income_growth"] = result["income_growth"] * 100
result["tfp_growth"] = result["tfp_growth"] * 100
result["capital_deepening"] = result["capital_deepening"] * 100

print(result.head())
result["tfp_share"] = result["tfp_growth"] / result["income_growth"]

print(result.head())
result = result[["country", "income_growth", "tfp_growth", "capital_deepening", "tfp_share", "alpha_k"]].copy()

result = result.rename(columns={
    "income_growth": "Income Growth Rate (%)",
    "tfp_growth": "TFP Growth Rate (%)",
    "capital_deepening": "Capital Deepening",
    "tfp_share": "TFP Share",
    "alpha_k": "Capital Share"
})

print(result.head())
result = result.round(2)
print(result.head())
oecd_avg = pd.DataFrame([{
    "country": "OECD Average",
    "Income Growth Rate (%)": result["Income Growth Rate (%)"].mean(),
    "TFP Growth Rate (%)": result["TFP Growth Rate (%)"].mean(),
    "Capital Deepening": result["Capital Deepening"].mean(),
    "TFP Share": result["TFP Share"].mean(),
    "Capital Share": result["Capital Share"].mean()
}])

oecd_avg = oecd_avg.round(2)

result_final = pd.concat([result, oecd_avg], ignore_index=True)

print(result_final.tail())
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

print(result_final)
result_final.to_excel("/content/oecd_growth_accounting_result.xlsx", index=False)

Saving pwt90.xlsx to pwt90 (21).xlsx
  countrycode country   currency_unit  year  rgdpe  rgdpo  pop  emp  avh  hc  \
0         ABW   Aruba  Aruban Guilder  1950    NaN    NaN  NaN  NaN  NaN NaN   
1         ABW   Aruba  Aruban Guilder  1951    NaN    NaN  NaN  NaN  NaN NaN   
2         ABW   Aruba  Aruban Guilder  1952    NaN    NaN  NaN  NaN  NaN NaN   
3         ABW   Aruba  Aruban Guilder  1953    NaN    NaN  NaN  NaN  NaN NaN   
4         ABW   Aruba  Aruban Guilder  1954    NaN    NaN  NaN  NaN  NaN NaN   

   ...  csh_g  csh_x  csh_m  csh_r  pl_c  pl_i  pl_g  pl_x  pl_m  pl_k  
0  ...    NaN    NaN    NaN    NaN   NaN   NaN   NaN   NaN   NaN   NaN  
1  ...    NaN    NaN    NaN    NaN   NaN   NaN   NaN   NaN   NaN   NaN  
2  ...    NaN    NaN    NaN    NaN   NaN   NaN   NaN   NaN   NaN   NaN  
3  ...    NaN    NaN    NaN    NaN   NaN   NaN   NaN   NaN   NaN   NaN  
4  ...    NaN    NaN    NaN    NaN   NaN   NaN   NaN   NaN   NaN   NaN  

[5 rows x 47 columns]
['countrycode', 'coun